# AME 5003 — Principles of NLP
## Lab 1: From Raw Text to a Working Text-Processing Pipeline

**Manipal School of Information Sciences (MSIS)**  
**Manipal Academy of Higher Education (MAHE)**

### What you will build
By the end of this lab, you will build a small processor for realistic student-service messages that can:

- extract structured fields using regular expressions;
- tokenize text and inspect boundary errors;
- normalize selected surface variations;
- make task-aware stop-word decisions;
- compare stemming and lemmatization;
- combine these steps into an end-to-end text-processing pipeline.

### Lab method
**Observe → Predict → Try → Test → Break → Improve → Explain**

You may use documentation, web search, Stack Overflow, or AI tools. However, for every major task you must:
1. write your prediction/approach first;
2. test the solution on the supplied examples;
3. find at least one case where it fails;
4. explain one assumption made by your solution.

> **Important:** The goal is not merely to make code run. The goal is to understand what information each processing decision preserves or destroys.


> **Dataset note:** The MSIS/MAHE-style messages, IDs and email addresses used here are illustrative and fictional; they are designed only for teaching text-processing decisions.


## Part 0 — Before code: what is the computer supposed to see?
**Suggested time: 0–15 min**

Read the following message:

> **URGENT!!!** Fee of **Rs. 2,450** was paid on **21/08/2026**.  
> The amount is **not reflected yet**.  
> Ref: **MSIS/26/104**  
> Contact: **student23@learner.manipal.edu**

### Activity 0.1 — Human extraction
Without writing code, identify:

| Field | Your answer |
|---|---|
| Urgency |  |
| Amount |  |
| Date |  |
| Reference ID |  |
| Email |  |
| Complaint meaning / intent |  |

### Activity 0.2 — Method choice
For each field above, decide whether you would begin with:

- **Regex / pattern rules**
- **Language-aware NLP / ML**
- **A hybrid of both**

Write one sentence explaining your choice.

### Checkpoint
Be ready to answer:

> **What kind of information is naturally represented by stable character patterns, and what kind depends on meaning/context?**


### Environment note
The notebook uses only lightweight CPU-friendly tools. Python's built-in `re` module is enough for the first half. Later sections use NLTK. In Google Colab NLTK is usually available; if your environment does not have it, install it with `pip install nltk`.

### Load the common lab dataset

In [ ]:

messages = [
    "URGENT!!! Fee of Rs. 2,450 was paid on 21/08/2026. The amount is not reflected yet. Ref: MSIS/26/104. Contact: student23@learner.manipal.edu",
    "Payment received: ₹3500 on 20-08-2026. Ref MSIS/FIN/2108. Mail: accounts.msis@manipal.edu",
    "Fee INR 12750 paid yesterday. Transaction ID: MAHE/PG/778. Email: student.one@gmail.com",
    "I have NOT received my refund of INR 2450. Please contact me at user24@learner.manipal.edu",
    "Registration completed successfully. Ref: MSIS/REG/4412",
    "My exam fee of Rs.2450 is still pending even though I paid on 19/08/2026.",
    "Approx amount: two thousand rupees. Payment is not showing.",
    "Hostel payment ₹2,450.50 received. Ref: MAHE/HST/219.",
    "Fee payment ₹2.5k is pending.",
    "Please update attendance for NLP. Student ID: MSISPG2026-041.",
    "Meeting rescheduled to 3.30 p.m. on 22/08/2026. Contact: faculty.msis@manipal.edu",
    "Good service!!! Really GOOD. Refund was processed quickly."
]

print("Number of messages:", len(messages))
for i, m in enumerate(messages[:3], 1):
    print(f"\n{i}. {m}")


Number of messages: 12

1. URGENT!!! Fee of Rs. 2,450 was paid on 21/08/2026. The amount is not reflected yet. Ref: MSIS/26/104. Contact: student23@learner.manipal.edu

2. Payment received: ₹3500 on 20-08-2026. Ref MSIS/FIN/2108. Mail: accounts.msis@manipal.edu

3. Fee INR 12750 paid yesterday. Transaction ID: MAHE/PG/778. Email: student.one@gmail.com


## Part 1 — Regex from scratch
**Suggested time: 15–45 min**

Regular expressions describe **surface patterns** in text.

### Activity 1.1 — Predict before running
For each pattern below, write what you expect it to match:

- `r"\d"`
- `r"\d+"`
- `r"\d{2}-\d{2}-\d{4}"`
- `r"[A-Z]+"`

Then test your prediction on one message from `messages`.

### Activity 1.2 — Build a date extractor
Your extractor should first handle:

- `21-08-2026`
- `21/08/2026`

Then test whether it handles:

- `21-08-26`
- `3.30 p.m.`
- `99-99-2026`

Before changing your pattern, write:

> **What date formats does my application actually need to accept?**

### TODO
Create `date_pattern` and test it.


In [ ]:

import re

sample = messages[0]

# TODO 1.1: try the four simple patterns from the activity.
# Example structure:
pattern = r"\d{2}[-/]\d{2}[-/]\d{4}"
print(re.findall(pattern, sample))


# # TODO 1.2: write a date pattern for DD-MM-YYYY and DD/MM/YYYY.
date_pattern = r"\d{2}[-/]\d{2}[-/](\d{4}|\d{2})"

date_tests = [
    "Date: 21-08-2026",
    "Date: 21/08/2026",
    "Date: 21-08-26",
    "Invalid-looking date: 99-99-2026",
]

for text in date_tests:
    match = re.search(date_pattern, text) if date_pattern else None
    print(text, "->", match.group(0) if match else None)


['21/08/2026']
Date: 21-08-2026 -> 21-08-2026
Date: 21/08/2026 -> 21/08/2026
Date: 21-08-26 -> 21-08-26
Invalid-looking date: 99-99-2026 -> 99-99-2026


## Part 2 — From matching to structured information
**Suggested time: 45–70 min**

Consider the amount forms:

- `₹2450`
- `₹ 2,450`
- `Rs.2450`
- `Rs. 2,450`
- `INR 2450`

### Activity 2.1 — Build an amount extractor
Design a pattern that captures:

1. a currency marker;
2. the numeric amount.

Use **capture groups** so you can inspect the pieces separately.

### Activity 2.2 — Normalize the captured value
Convert:

`Rs. 2,450`

into a structured representation such as:

```python
{"currency": "INR", "amount": 2450}
```

### Think
Why is `2450` more useful computationally than the string `"Rs. 2,450"`?

What information might you still want to retain from the raw text?


In [ ]:

amount_tests = [
    "Paid ₹2450 today",
    "Fee: ₹ 2,450",
    "Paid Rsp 2450",
    "Paid Rs. 2,450",
    "Paid INR 2450",
]

# TODO 2.1: write an amount pattern with capture groups.
amount_pattern = r"(₹|Rs\.?|Rsp|INR)\s*([\d,]+(?:\.\d{1,2})?)"
#here \. is Rs. should be the substring
for text in amount_tests:
    m = re.search(amount_pattern, text) if amount_pattern else None
    if m:
        print("TEXT:", text)
        print(" full match:", m.group(0))
        print(" groups:", m.groups())
    else:
        print("NO MATCH:", text)


# TODO 2.2: write a function that converts a matched amount to structured data.
def normalize_amount(match):
    currency = match.group(1)
    amount = match.group(2)

    # Normalize currency names
    if currency in ("Rs", "Rs.", "Rsp"):
        currency = "INR"
    elif currency == "₹":
        currency = "INR"

    return {
        "currency": currency,
        "amount": int(amount.replace(",", ""))
    }



TEXT: Paid ₹2450 today
 full match: ₹2450
 groups: ('₹', '2450')
TEXT: Fee: ₹ 2,450
 full match: ₹ 2,450
 groups: ('₹', '2,450')
TEXT: Paid Rsp 2450
 full match: Rsp 2450
 groups: ('Rsp', '2450')
TEXT: Paid Rs. 2,450
 full match: Rs. 2,450
 groups: ('Rs.', '2,450')
TEXT: Paid INR 2450
 full match: INR 2450
 groups: ('INR', '2450')


## Part 3 — Break your own regex
**Suggested time: 70–90 min**

A robust NLP workflow does not stop after the first successful match.

### Activity 3.1 — Test difficult cases

| Input | Should your system extract an amount? | Actual result | Correct? |
|---|---:|---|---:|
| `₹2,450` | Yes |  |  |
| `Rs.2450` | Yes |  |  |
| `INR 2450` | Yes |  |  |
| `₹2,450.50` | Yes |  |  |
| `₹2.5k` | Maybe, depending on specification |  |  |
| `two thousand rupees` | Yes semantically; difficult for simple regex |  |  |

### Activity 3.2 — False positive / false negative
Find:

- **one false positive** produced by one of your regexes;
- **one false negative** produced by one of your regexes.

### Required reflection
Complete:

> My regex assumes that ____________________.  
> It fails when ____________________.  
> I would improve it by ____________________.

### Key conceptual test
Does matching `99-99-2026` prove that it is a valid date? Explain.


In [ ]:

hard_amounts = [
    "₹2,450",
    "Rs.2450",
    "INR 2450",
    "₹2,450.50",
    "₹2.5k",
    "two thousand rupees",
]

# TODO 3.1: test your current amount_pattern on all cases.
for text in hard_amounts:
    m = re.search(amount_pattern, text) if amount_pattern else None
    print(text, "->", m.group(0) if m else None)

# TODO 3.2: add at least one deliberately difficult input of your own.
my_break_case = "Paid Rs 1,000/- yesterday"
print("My break case:", my_break_case)


₹2,450 -> ₹2,450
Rs.2450 -> Rs.2450
INR 2450 -> INR 2450
₹2,450.50 -> ₹2,450.50
₹2.5k -> ₹2.5
two thousand rupees -> None
My break case: Paid Rs 1,000/- yesterday


---

## 🛑 STOP POINT 1 — Instructor checkpoint

Before continuing, show or discuss:

1. one regex that works on your intended format;
2. one false positive or false negative;
3. one assumption made by your regex.

**Do not continue until the instructor releases the next part.**

---


## Part 4 — Tokenization: why `split()` is not enough
**Suggested time: 90–115 min**

Use this text:

> `MSIS-based NLP models don't always handle ₹1,25,000 invoices or e-mail IDs like dean-office@manipal.edu.`

### Activity 4.1 — Naive tokenization
First try Python's whitespace splitting.

Then inspect these items:

- `MSIS-based`
- `don't`
- `₹1,25,000`
- `e-mail`
- `dean-office@manipal.edu`

### Activity 4.2 — Decide the boundaries
For each item, decide whether it should be:

- one token;
- multiple tokens;
- extracted as a structured field before ordinary tokenization.

There may be more than one defensible answer.

### Required explanation
State the downstream application you assumed, for example:

- information extraction;
- search;
- sentiment analysis;
- POS tagging.

Then justify your token boundaries.


In [ ]:

token_text = "MSIS-based NLP models don't always handle ₹1,25,000 invoices or e-mail IDs like dean-office@manipal.edu."

# TODO 4.1: run whitespace tokenization.
whitespace_tokens = token_text.split()
print(whitespace_tokens)

# TODO 4.2: design a better tokenization approach.
# You may use regex, NLTK, spaCy, documentation, or AI assistance,
# but explain what your tokenizer does with the five difficult items.
token_pattern = r"""
    [\w.+-]+@[\w.-]+\.\w+     # email address
    | ₹[\d,]+(?:\.\d+)?        # rupee amount
    | [A-Za-z]+(?:-[A-Za-z]+)+  # hyphenated words
    | [A-Za-z]+(?:'[A-Za-z]+)?  # words and contractions
    | \d+                       # standalone numbers
    | [^\s\w]                   # punctuation/symbols
"""
better_tokens = re.findall(token_pattern, token_text, re.VERBOSE)
print(better_tokens)


['MSIS-based', 'NLP', 'models', "don't", 'always', 'handle', '₹1,25,000', 'invoices', 'or', 'e-mail', 'IDs', 'like', 'dean-office@manipal.edu.']
['MSIS-based', 'NLP', 'models', "don't", 'always', 'handle', '₹1,25,000', 'invoices', 'or', 'e-mail', 'IDs', 'like', 'dean-office@manipal.edu', '.']


## Part 5 — Normalization: change or preserve?
**Suggested time: 115–135 min**

Normalization makes selected variants look alike. But every transformation may remove information.

### Activity 5.1 — Compare these cases

1. `The US team visited us.`
2. `Good service!!! Really GOOD.`
3. `Fee: Rs. 2,450`
4. `Fee: INR 2450`
5. `Fee: ₹2450`

For each case decide:

- lowercase?
- normalize spacing?
- normalize currency representation?
- remove punctuation?
- preserve emphasis/capitalization?

### Activity 5.2 — Build a small normalization function
Your function should **not** blindly do every possible cleaning operation.

State your assumed application and implement only justified transformations.

### Required reflection
> One normalization decision I would *not* apply universally is __________ because __________.


In [ ]:

normalization_tests = [
    "The US team visited us.",
    "Good service!!! Really GOOD.",
    "Fee: Rs. 2,450",
    "Fee: INR 2450",
    "Fee: ₹2450",
]

def normalize_text(text):
    # TODO: choose and justify the transformations you want.
    text = re.sub(r"₹|Rs\.?|Rsp|INR", " INR ", text, flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?])", r"\1", text)

    return text.strip()
    return text

for text in normalization_tests:
    print("RAW :", text)
    print("NORM:", normalize_text(text))
    print()


RAW : The US team visited us.
NORM: the us team visited us.

RAW : Good service!!! Really GOOD.
NORM: good service! really good.

RAW : Fee: Rs. 2,450
NORM: fee: inr 2,450

RAW : Fee: INR 2450
NORM: fee: inr 2450

RAW : Fee: ₹2450
NORM: fee: inr 2450



## Part 6 — Stop words: test the rule instead of memorizing it
**Suggested time: 135–150 min**

Consider:

1. `This movie is not good.`
2. `The patient has no fever.`
3. `The payment is not reflected.`
4. `I am not unhappy with the result.`

### Activity 6.1 — Use a standard stop-word list
Remove stop words and inspect what remains.

### Activity 6.2 — Diagnose semantic damage
Identify any removed word that changes the meaning strongly.

### Activity 6.3 — Build a task-aware list
Modify the stop-word decision so that important negation words are preserved.

### Question
Is a word a “stop word” because it is unimportant in language, or because it is unhelpful **for a particular task**?


In [ ]:

# Setup for NLTK resources. In Colab this should run once.
import nltk
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

negation_tests = [
    "This movie is not good.",
    "The patient has no fever.",
    "The payment is not reflected.",
    "I am not unhappy with the result.",
]

# TODO 6.3: preserve critical negation words
custom_stopwords = set(english_stopwords)
custom_stopwords.discard("not")
custom_stopwords.discard("no")

for text in negation_tests:
    print("\nTEXT:", text)

    # TODO 6.1: simple tokenization
    tokens = text.lower().split()

    # Standard stopword removal
    standard_filtered = [
        token for token in tokens
        if token not in english_stopwords
    ]

    # Task-aware stopword removal
    custom_filtered = [
        token for token in tokens
        if token not in custom_stopwords
    ]

    print("Tokens:", tokens)
    print("Standard:", standard_filtered)
    print("Custom  :", custom_filtered)

    # TODO 6.2
    print("'not' removed:", "not" not in standard_filtered)
    print("'no' removed :", "no" not in standard_filtered)



TEXT: This movie is not good.
Tokens: ['this', 'movie', 'is', 'not', 'good.']
Standard: ['movie', 'good.']
Custom  : ['movie', 'not', 'good.']
'not' removed: True
'no' removed : True

TEXT: The patient has no fever.
Tokens: ['the', 'patient', 'has', 'no', 'fever.']
Standard: ['patient', 'fever.']
Custom  : ['patient', 'no', 'fever.']
'not' removed: True
'no' removed : True

TEXT: The payment is not reflected.
Tokens: ['the', 'payment', 'is', 'not', 'reflected.']
Standard: ['payment', 'reflected.']
Custom  : ['payment', 'not', 'reflected.']
'not' removed: True
'no' removed : True

TEXT: I am not unhappy with the result.
Tokens: ['i', 'am', 'not', 'unhappy', 'with', 'the', 'result.']
Standard: ['unhappy', 'result.']
Custom  : ['not', 'unhappy', 'result.']
'not' removed: True
'no' removed : True


## Part 7 — Stemming: useful collapse or information damage?
**Suggested time: 150–160 min**

Use the words:

`connect, connects, connected, connecting, connection, connectivity, studies, studying, university, universal, policy, police`

### Activity 7.1
Use the Porter stemmer and record the outputs.

### Activity 7.2
Identify:

- one useful collapse;
- one potentially risky collapse;
- one example where a stem is not a normal dictionary word.

### Required explanation
Why can stemming reduce **feature fragmentation** in sparse bag-of-words style representations?


In [ ]:

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stem_words = [
    "connect", "connects", "connected", "connecting",
    "connection", "connectivity",
    "studies", "studying",
    "university", "universal",
    "policy", "police"
]

# TODO 7.1: print word -> stem for every word.
for word in stem_words:
    print(word, "->", stemmer.stem(word))


connect -> connect
connects -> connect
connected -> connect
connecting -> connect
connection -> connect
connectivity -> connect
studies -> studi
studying -> studi
university -> univers
universal -> univers
policy -> polici
police -> polic


## Part 8 — Lemmatization: dictionary form with linguistic knowledge
**Suggested time: 160–170 min**

Compare lemmatization with stemming.

Suggested words in context:

- `studies`
- `was`
- `better`
- `saw` (noun: a cutting tool)
- `saw` (verb: past tense of *see*)
- `running`

### Activity 8.1
Use a lemmatizer and compare the output with your stems.

### Activity 8.2
Investigate whether supplying a POS category changes any output.

You may use web/AI to find how the chosen library expects POS information.

### Bridge question
Why might lemmatization need POS tagging?


In [ ]:

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

lemma_words = ["studies", "was", "better", "saw", "running"]

# TODO 8.1: lemmatize without supplying POS
print("Without POS:")
for word in lemma_words:
    print(word, "->", lemmatizer.lemmatize(word))

# TODO 8.2: try appropriate POS arguments
print("\nWith POS:")
pos_examples = {
    "studies": "v",
    "was": "v",
    "better": "a",
    "saw": "v",
    "running": "v"
}

for word, pos in pos_examples.items():
    print(word, "->", lemmatizer.lemmatize(word, pos=pos))


Without POS:
studies -> study
was -> wa
better -> better
saw -> saw
running -> running

With POS:
studies -> study
was -> be
better -> good
saw -> saw
running -> run


---

## 🛑 STOP POINT 2 — Class comparison

Prepare to answer:

1. Which transformation in Parts 4–8 was most dangerous if applied blindly?
2. Which example changed your mind about “standard preprocessing”?
3. Give one situation where you would keep the raw text alongside the processed form.

---


## Part 9 — End-to-end real-world challenge
**Suggested time: 170–178 min**

You now have the pieces. Build a small processor for the MSIS/MAHE student-service messages.

### Minimum requirements
Your function should attempt to return:

- amount (normalized numeric value where possible);
- date;
- reference ID;
- email;
- tokens;
- normalized tokens/text;
- a task-aware stop-word output;
- stems **or** lemmas (state which you chose and why).

### Design freedom
You do **not** have to use every preprocessing step.

You must justify:

1. what you extract before tokenization;
2. what you normalize;
3. what you deliberately preserve;
4. whether you use stemming, lemmatization, or neither;
5. one known failure of your final pipeline.

### Deliverable
Run the function on at least **five** supplied messages, including one difficult case.


In [ ]:

def process_message(text):
    result = {
        "raw_text": text,
        "amount": None,
        "date": None,
        "reference": None,
        "email": None,
        "tokens": None,
        "normalized": None,
        "stopword_filtered": None,
        "reduced_forms": None,
    }

    # TODO: regex extraction

    # Amount
    amount_match = re.search(
        r"(₹|Rs\.?|Rsp|INR)\s*([\d,]+(?:\.\d{1,2})?)",
        text,
        re.IGNORECASE
    )

    if amount_match:
        result["amount"] = {
            "currency": "INR",
            "value": float(amount_match.group(2).replace(",", ""))
        }

    # Date: simple dd/mm/yyyy or dd-mm-yyyy
    date_match = re.search(
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        text
    )
    if date_match:
        result["date"] = date_match.group(0)

    # Reference number
    reference_match = re.search(
        r"\b(?:ref(?:erence)?|txn|transaction)[\s:#-]*([A-Za-z0-9-]+)",
        text,
        re.IGNORECASE
    )
    if reference_match:
        result["reference"] = reference_match.group(1)

    # Email
    email_match = re.search(
        r"\b[\w.+-]+@[\w.-]+\.\w+\b",
        text
    )
    if email_match:
        result["email"] = email_match.group(0)

    # TODO: tokenization
    token_pattern = r"""
        [\w.+-]+@[\w.-]+\.\w+       # email
        | ₹[\d,]+(?:\.\d+)?         # rupee amount
        | [A-Za-z]+(?:-[A-Za-z]+)+   # hyphenated word
        | [A-Za-z]+(?:'[A-Za-z]+)?   # word/contraction
        | \d+(?:[.,]\d+)?           # number
        | [^\s\w]                   # punctuation
    """

    tokens = re.findall(token_pattern, text, re.VERBOSE)
    result["tokens"] = tokens

    # TODO: normalization
    normalized = text

    normalized = re.sub(
        r"₹|Rs\.?|Rsp|INR",
        " INR ",
        normalized,
        flags=re.IGNORECASE
    )

    normalized = normalized.lower()
    normalized = re.sub(r"!+", "!", normalized)
    normalized = re.sub(r"\s+", " ", normalized)
    normalized = re.sub(r"\s+([.,!?])", r"\1", normalized)

    result["normalized"] = normalized.strip()

    # TODO: stop-word decision
    # Preserve "not" and "no" because they affect meaning.
    custom_stopwords = set(english_stopwords)
    custom_stopwords.discard("not")
    custom_stopwords.discard("no")

    normalized_tokens = normalized.split()

    result["stopword_filtered"] = [
        token for token in normalized_tokens
        if token not in custom_stopwords
    ]

    # TODO: stemming OR lemmatization
    # Use stemming here because it does not require POS tagging.
    result["reduced_forms"] = [
        stemmer.stem(token)
        for token in result["stopword_filtered"]
    ]

    return result


# Test on at least five messages.
for m in messages[:5]:
    print(process_message(m))
    print("-" * 80)


{'raw_text': 'URGENT!!! Fee of Rs. 2,450 was paid on 21/08/2026. The amount is not reflected yet. Ref: MSIS/26/104. Contact: student23@learner.manipal.edu', 'amount': {'currency': 'INR', 'value': 2450.0}, 'date': '21/08/2026', 'reference': 'lected', 'email': 'student23@learner.manipal.edu', 'tokens': ['URGENT', '!', '!', '!', 'Fee', 'of', 'Rs', '.', '2,450', 'was', 'paid', 'on', '21', '/', '08', '/', '2026', '.', 'The', 'amount', 'is', 'not', 'reflected', 'yet', '.', 'Ref', ':', 'MSIS', '/', '26', '/', '104', '.', 'Contact', ':', 'student23@learner.manipal.edu'], 'normalized': 'urgent! fee of inr 2,450 was paid on 21/08/2026. the amount is not reflected yet. ref: msis/26/104. contact: student23@learner.manipal.edu', 'stopword_filtered': ['urgent!', 'fee', 'inr', '2,450', 'paid', '21/08/2026.', 'amount', 'not', 'reflected', 'yet.', 'ref:', 'msis/26/104.', 'contact:', 'student23@learner.manipal.edu'], 'reduced_forms': ['urgent!', 'fee', 'inr', '2,450', 'paid', '21/08/2026.', 'amount', 'n

## Part 10 — Reflection and mini-viva
**Suggested time: 178–180 min + submission**

Answer briefly:

1. Which field in your final pipeline was easiest to extract reliably? Why?
2. Give one false positive or false negative from your work.
3. Why is whitespace splitting a weak general tokenizer?
4. Give one case where lowercasing may be harmful.
5. Why can stop-word removal damage a sentiment/complaint application?
6. State one precise difference between stemming and lemmatization.
7. Which preprocessing step in your pipeline is most likely to damage useful information?
8. What did an AI/web suggestion get wrong, oversimplify, or leave unstated?

### Submission checklist
- [ ] Predictions/approaches were written before major implementations.
- [ ] Regex extraction tested on difficult examples.
- [ ] At least one false positive/false negative documented.
- [ ] Tokenization decision justified by downstream task.
- [ ] Normalization decision justified.
- [ ] Stop-word decision tested on negation.
- [ ] Stemming and lemmatization compared.
- [ ] End-to-end function run on ≥5 messages.
- [ ] One known limitation clearly stated.
